# Tests for ObserveClient

In [ ]:
#|default_exp observe.test_client

In [ ]:
#|export
import pytest

from netrun.net._net._net import Net
from netrun.net.config import (
    NetConfig,
    GraphConfig,
    NodeConfig,
    PortConfig,
    EdgeConfig,
    PoolConfig,
    MainPoolConfig,
    NodeExecutionConfig,
)

from netrun_utils.observe.server import ObserveServer
from netrun_utils.observe.client import ObserveClient

## Helpers

In [ ]:
#|export
def _make_config() -> NetConfig:
    return NetConfig(
        pools={"main": PoolConfig(spec=MainPoolConfig())},
        graph=GraphConfig(
            nodes=[
                NodeConfig(
                    name="a",
                    out_ports={"out": PortConfig()},
                    execution_config=NodeExecutionConfig(
                        pools=["main"],
                        exec_node_func=lambda ctx, packets: None,
                    ),
                ),
                NodeConfig(
                    name="b",
                    in_ports={"inp": PortConfig()},
                    execution_config=NodeExecutionConfig(
                        pools=["main"],
                        exec_node_func=lambda ctx, packets: None,
                    ),
                ),
            ],
            edges=[
                EdgeConfig(source_node="a", source_port="out", target_node="b", target_port="inp"),
            ],
        ),
        retain_epoch_logs=True,
    )

## Client Tests

In [ ]:
#|export
async def test_client_get_status():
    config = _make_config()
    async with Net(config, run_source_nodes=False) as net:
        async with ObserveServer(net, port=18322) as server:
            async with ObserveClient(server.url) as client:
                status = await client.get_status()
                assert status.started is True
                assert set(status.node_names) == {"a", "b"}


async def test_client_get_nodes():
    config = _make_config()
    async with Net(config, run_source_nodes=False) as net:
        async with ObserveServer(net, port=18323) as server:
            async with ObserveClient(server.url) as client:
                nodes = await client.get_nodes()
                assert len(nodes) == 2

                node_a = await client.get_node("a")
                assert node_a.name == "a"
                assert node_a.out_port_names == ["out"]


async def test_client_get_edges():
    config = _make_config()
    async with Net(config, run_source_nodes=False) as net:
        async with ObserveServer(net, port=18324) as server:
            async with ObserveClient(server.url) as client:
                edges = await client.get_edges()
                assert len(edges) == 1
                assert edges[0].source_node == "a"
                assert edges[0].target_node == "b"


async def test_client_enable_disable():
    config = _make_config()
    async with Net(config, run_source_nodes=False) as net:
        async with ObserveServer(net, port=18325) as server:
            async with ObserveClient(server.url) as client:
                resp = await client.disable_node("b")
                assert resp.ok is True

                node_b = await client.get_node("b")
                assert node_b.enabled is False

                resp = await client.enable_node("b")
                assert resp.ok is True


async def test_client_inject_data():
    config = _make_config()
    async with Net(config, run_source_nodes=False) as net:
        async with ObserveServer(net, port=18326) as server:
            async with ObserveClient(server.url) as client:
                resp = await client.inject_data("b", "inp", [1, 2])
                assert resp.ok is True